# JERICHON — AFML 04: Исправленный nested Optuna + исполнение по reverse-сигналу

**Что исправлено относительно AFML 03:**
1. `Best trial value` — это score внутренней оптимизации, а не итоговая доходность внешнего OOS-бэктеста. Внешние периоды специально не используются для выбора trial.
2. В каждой trial конфигурации `dollar_threshold` реально перестраиваются Dollar Bars и все 474 признака. Порог выбирается из небольшого дискретного набора, чтобы один и тот же набор признаков строился максимум один раз за запуск и кэшировался.
3. Бэктест закрывает открытую позицию при противоположном уверенном сигнале: сигнал известен на закрытии бара, исполнение — на открытии следующего бара.
4. В аудит добавлены фактические относительные расстояния TP/SL. `tp_mult=1.7` — это множитель target-volatility, а НЕ TP в 0.17%, 0.6% или 1.7%. Реальный процент зависит от target, поэтому его надо проверять по сделкам.
5. Внешние WF-границы фиксируются по исходной временной шкале, чтобы изменение Dollar Bars не двигало сами OOS-периоды.

**Важно:** модель Random Forest фиксирована; Optuna не оптимизирует число деревьев или глубину. Это исследовательский эксперимент на короткой истории, не гарантия прибыльности.


## 1. Подготовка локальных данных

Используется локальная история AlgoPack. В исходной части notebook признаки строятся на исходном `DOLLAR_THRESHOLD`, чтобы проверить доступность pipeline. Затем функция `build_feature_context_for_threshold()` полностью перестраивает Dollar Bars и 474 признака для каждого порога из `DOLLAR_THRESHOLD_CANDIDATES`.

**Почему предыдущий прогон был слишком быстрым:** он строил Dollar Bars и признаки один раз до Optuna, а в trial менял только CUSUM/Triple Barrier параметры. Поэтому `dollar_threshold` в действительности не оптимизировался. В этой версии разные пороги действительно пересобирают данные. Кэш ограничивает повторные перестройки одним разом на каждый кандидат.


In [44]:
\
# 0. CONFIG + IMPORTS
from pathlib import Path
import os, sys, inspect, warnings, json, math, importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mplcyberpunk
plt.style.use("cyberpunk")
import re
from sklearn.impute import SimpleImputer

TICKER = "VTBR"
DOLLAR_THRESHOLD = 10_000_000
MSK = "Europe/Moscow"
CUSUM_THRESHOLD = 0.001
PT_SL = (1.0, 1.0)
VERTICAL_HORIZON = 50
TARGET_WINDOW = 55
EXPECTED_FEATURES = 474

# Автоматически ищем корень репозитория JERICHON, не фиксируем путь конкретного пользователя.
def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for p in candidates:
        if (p / "data" / "for_ml" / "storage" / "history").exists() and (p / "src" / "data" / "builders").exists():
            return p
    raise FileNotFoundError(
        "Не найден корень JERICHON. Открой notebook из папки репозитория "
        "или укажи PROJECT_ROOT вручную."
    )

# PROJECT_ROOT may be set explicitly; otherwise discover the repo from cwd/parents.
def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for p in candidates:
        if (p / "data" / "for_ml" / "storage" / "history").exists() and (p / "src" / "data" / "builders").exists():
            return p
    raise FileNotFoundError(
        "Не найден корень JERICHON. Открой notebook из папки репозитория "
        "или укажи PROJECT_ROOT вручную."
    )

ROOT = Path(r"C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\data")
if str(ROOT.parent) not in sys.path:
    sys.path.insert(0, str(ROOT.parent))
HISTORY_ROOT = ROOT / "for_ml" / "storage" / "history"

from src.data.builders.dollar_bars import InformationDrivenBars
from src.data.builders.micro_features import MicroFeatures
from src.data.builders.build_features import FeatureBuilder
from src.data.builders import jerichon_feature_pipeline as jfp

print("Ticker       :", TICKER)
print("Data root    :", ROOT)
print("History root :", HISTORY_ROOT)
print("Pipeline     :", Path(inspect.getfile(jfp)).resolve())
print("Pipeline API :", [n for n in (
    "build_feature_dataset", "build_dollar_bars_from_trades", "_base",
    "_build_sc_features", "_choose_availability", "_attach_source"
) if hasattr(jfp, n)])


Ticker       : VTBR
Data root    : C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\data
History root : C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\data\for_ml\storage\history
Pipeline     : C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\data\for_ml\builders\jerichon_feature_pipeline.py
Pipeline API : ['build_feature_dataset', 'build_dollar_bars_from_trades', '_base', '_build_sc_features', '_choose_availability', '_attach_source']


## 1. Загрузка локальной истории

Формат путей сохранён **как в `jerichon_mark_63(3).ipynb`**:

- `data/for_ml/storage/history/realtime/{TICKER}/{YYYY-MM-DD}/trades.parquet`
- `data/for_ml/storage/history/supercandles/{TICKER}/{YYYY-MM-DD}/{tradestats,obstats,orderstats}.parquet`

Файлы только читаются. Если какой-то источник отсутствует, notebook покажет это явно.

In [45]:
def available_dates(history_root, ticker):
    dates = set()
    for kind in ("realtime", "supercandles"):
        p = history_root / kind / ticker
        if p.exists():
            dates.update(x.name for x in p.iterdir() if x.is_dir())
    return sorted(d for d in dates if len(d) == 10 and d[4] == "-" and d[7] == "-")

def load_local_history(history_root, ticker):
    keys = ("candles", "trades", "orderbook", "tradestats", "obstats", "orderstats")
    frames = {k: [] for k in keys}
    dates = available_dates(history_root, ticker)

    if not dates:
        raise FileNotFoundError(f"Для {ticker} не найдены сохранённые даты в {history_root}")

    for day in dates:
        for key in ("candles", "trades", "orderbook"):
            p = history_root / "realtime" / ticker / day / f"{key}.parquet"
            if p.exists():
                try:
                    x = pd.read_parquet(p)
                    if not x.empty:
                        x = x.copy()
                        x["file_date"] = day
                        frames[key].append(x)
                except Exception as e:
                    print(f"WARNING: не удалось прочитать {p}: {e}")

        for key in ("tradestats", "obstats", "orderstats"):
            p = history_root / "supercandles" / ticker / day / f"{key}.parquet"
            if p.exists():
                try:
                    x = pd.read_parquet(p)
                    if not x.empty:
                        x = x.copy()
                        x["file_date"] = day
                        frames[key].append(x)
                except Exception as e:
                    print(f"WARNING: не удалось прочитать {p}: {e}")

    out = {k: (pd.concat(v, ignore_index=True) if v else pd.DataFrame()) for k, v in frames.items()}
    print(f"Saved dates discovered: {len(dates)}")
    print(f"Date range            : {dates[0]} → {dates[-1]}")
    print("\\nLocal rows:")
    display(pd.DataFrame({"source": list(out), "rows": [len(out[k]) for k in out]}))
    missing = [s for s in ("trades", "tradestats", "obstats", "orderstats") if out[s].empty]
    if missing:
        raise RuntimeError(f"Не хватает локальных источников: {missing}. API-запрос не будет выполняться.")
    return out, dates

data, SAVED_DATES = load_local_history(HISTORY_ROOT, TICKER)

Saved dates discovered: 30
Date range            : 2026-08-28 → 2026-10-15
\nLocal rows:


,source,rows
0,candles,27259
1,trades,2802985
2,orderbook,560
3,tradestats,5460
4,obstats,5327
5,orderstats,5502


## 2. Построение Dollar Bars и 474 признаков

Используем подготовленный модуль `jerichon_feature_pipeline.py`, а не копируем его инженерный код в notebook. Важно: **не вызываем** публичный `build_feature_dataset()`, если он сам создаёт `AlgoPackProvider`; вместо этого вызываем локальные функции модуля над уже прочитанными DataFrame.

Если локальная версия модуля не содержит ожидаемые helper-функции, выполнение остановится с понятной ошибкой, а не отправит запрос в API.

In [46]:
# Сначала проверяем интерфейс установленного файла — никаких сетевых вызовов здесь нет.
required_helpers = (
    "_trades_for_bars",
    "build_dollar_bars_from_trades",
    "_base",
    "_build_sc_features",
    "_choose_availability",
    "_attach_source",
)
missing_helpers = [name for name in required_helpers if not hasattr(jfp, name)]
if missing_helpers:
    raise RuntimeError(
        "В установленном jerichon_feature_pipeline.py не найдены helper-функции: "
        f"{missing_helpers}. Проверь, что в проекте стоит актуальная версия pipeline. "
        "Этот notebook намеренно не переключается на API."
    )

print("Helper signatures:")
for name in required_helpers:
    print(f"  {name}{inspect.signature(getattr(jfp, name))}")

Helper signatures:
  _trades_for_bars(trades)
  build_dollar_bars_from_trades(trades, threshold=13000000)
  _base(bars)
  _build_sc_features(raw_sources, bars_for_features)
  _choose_availability(df, source_name)
  _attach_source(bars, feature_frame, raw_source, source)


In [47]:
# Dollar Bars: используем тот же денежный объём VALUE / PRICE, что и в reference.
bars = jfp.build_dollar_bars_from_trades(data["trades"], threshold=DOLLAR_THRESHOLD)
if bars is None or bars.empty:
    raise RuntimeError("Dollar Bar builder вернул пустой результат.")

# Унифицируем имя времени для последующих causal joins.
if "bar_ts" not in bars.columns:
    if "close_time" not in bars.columns:
        raise RuntimeError(f"В bars нет bar_ts/close_time. Колонки: {list(bars.columns)}")
    bars["bar_ts"] = pd.to_datetime(bars["close_time"], errors="coerce", utc=True)
else:
    bars["bar_ts"] = pd.to_datetime(bars["bar_ts"], errors="coerce", utc=True)

bars = (bars.dropna(subset=["bar_ts"])
             .sort_values("bar_ts")
             .drop_duplicates("bar_ts", keep="last")
             .reset_index(drop=True))

print("Dollar Bars:", len(bars))
print("Period:", bars["bar_ts"].min(), "→", bars["bar_ts"].max())
display(bars.head(3))

Dollar Bars: 8634
Period: 2026-08-28 04:02:32+00:00 → 2026-10-15 20:37:59+00:00


,open,high,low,close,volume,value,count,open_time,close_time,bar_ts
0,50.80,50.865,50.780,50.790,200076.007083,10166583.32,430,2026-08-28 03:59:45+00:00,2026-08-28 04:02:32+00:00,2026-08-28 04:02:32+00:00
1,50.79,50.810,50.650,50.665,197688.004437,10028117.75,282,2026-08-28 04:02:32+00:00,2026-08-28 04:05:04+00:00,2026-08-28 04:05:04+00:00
2,50.67,50.800,50.605,50.800,197585.005131,10011639.21,325,2026-08-28 04:05:04+00:00,2026-08-28 04:10:16+00:00,2026-08-28 04:10:16+00:00


In [48]:
# Base 73 + Super Candle 401. Работаем через helper-функции pipeline.
base_raw = jfp._base(bars.copy())
if isinstance(base_raw, tuple):
    # Поддержка версий helper, возвращающих (features, metadata).
    base_raw = base_raw[0]
if not isinstance(base_raw, pd.DataFrame):
    raise TypeError(f"_base должен вернуть DataFrame, получено: {type(base_raw)}")

# Приводим base frame к контракту bar_ts + 73 числовых признака.
if "bar_ts" not in base_raw.columns:
    if "close_time" in base_raw.columns:
        base_raw = base_raw.rename(columns={"close_time": "bar_ts"})
    elif isinstance(base_raw.index, pd.DatetimeIndex):
        base_raw = base_raw.reset_index().rename(columns={base_raw.index.name or "index": "bar_ts"})
    elif "ts" in base_raw.columns:
        base_raw = base_raw.rename(columns={"ts": "bar_ts"})
    else:
        raise RuntimeError(f"Не найден временной ключ в _base output. Колонки: {list(base_raw.columns)[:25]}")

base_raw["bar_ts"] = pd.to_datetime(base_raw["bar_ts"], errors="coerce", utc=True)
base_raw = base_raw.dropna(subset=["bar_ts"]).sort_values("bar_ts").drop_duplicates("bar_ts", keep="last").reset_index(drop=True)

# _build_sc_features может возвращать feature_frames либо (feature_frames, groups).
# Адаптируем Dollar Bars к формату, ожидаемому pipeline.
bars_for_sc = bars.copy()

# В pipeline ожидаются ts и CLOSE.
if "ts" not in bars_for_sc.columns:
    if "bar_ts" in bars_for_sc.columns:
        bars_for_sc["ts"] = bars_for_sc["bar_ts"]
    elif "close_time" in bars_for_sc.columns:
        bars_for_sc["ts"] = bars_for_sc["close_time"]
    else:
        raise KeyError("В bars отсутствует временная колонка ts/bar_ts/close_time")

if "CLOSE" not in bars_for_sc.columns:
    if "close" in bars_for_sc.columns:
        bars_for_sc["CLOSE"] = bars_for_sc["close"]
    else:
        raise KeyError("В bars отсутствует цена закрытия CLOSE/close")

bars_for_sc["ts"] = pd.to_datetime(bars_for_sc["ts"], errors="coerce", utc=True)

bars_for_sc = (
    bars_for_sc.dropna(subset=["ts"])
    .sort_values("ts")
    .reset_index(drop=True))

sc_result = jfp._build_sc_features(data, bars_for_sc)

if isinstance(sc_result, tuple):
    feature_frames = sc_result[0]
    feature_groups = sc_result[1] if len(sc_result) > 1 else {}
else:
    feature_frames = sc_result
    feature_groups = {}

if not isinstance(feature_frames, dict):
    raise TypeError("_build_sc_features должен возвращать dict feature frames.")

print("Base output shape:", base_raw.shape)
print("Super Candle feature groups:", list(feature_frames))

Base output shape: (8634, 74)
Super Candle feature groups: ['tradestats', 'orderstats', 'obstats']


C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\data\for_ml\builders\jerichon_feature_pipeline.py:77: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{c}_z{w}"] = (s - mean) / (std + 1e-12)
C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\data\for_ml\builders\jerichon_feature_pipeline.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{c}_ma{w}"] = mean
C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\data\for_ml\builders\jerichon_feature_pipeline.py:71: PerformanceWarning: DataFrame is highly fragmented.  Thi

In [49]:
# Availability validation and causal joins, source by source.
# Helpers' outputs are inspected to support the current prepared module contract.
validated = {}
availability_diagnostics = []

for source in ("tradestats", "obstats", "orderstats"):
    if data[source].empty:
        raise RuntimeError(f"Локальный источник {source} пуст.")

    result = jfp._choose_availability(data[source].copy(), source)
    if isinstance(result, tuple):
        validated[source] = result[0]
        if len(result) > 1 and isinstance(result[1], pd.DataFrame):
            availability_diagnostics.append(result[1])
    else:
        validated[source] = result

print("Validated sources:")
for source, frame in validated.items():
    print(f"  {source:12s}: {len(frame):,} rows, columns={len(frame.columns)}")

if availability_diagnostics:
    display(pd.concat(availability_diagnostics, ignore_index=True))

Validated sources:
  tradestats  : 5,460 rows, columns=2
  obstats     : 5,327 rows, columns=2
  orderstats  : 5,502 rows, columns=2


In [50]:
# Normalize the bars frame for the pipeline's attach helper.
bar_frame = bars.copy()
if "bar_ts" not in bar_frame.columns:
    raise RuntimeError("bar_ts missing from Dollar Bars.")

# Resolve each raw source to the corresponding feature frame by name.
def resolve_feature_frame(source, frames):
    if source in frames:
        return frames[source]
    matches = [k for k in frames if source.lower() in str(k).lower()]
    if len(matches) == 1:
        return frames[matches[0]]
    raise RuntimeError(f"Не удалось однозначно сопоставить {source} с feature frame. Доступны: {list(frames)}")

causal = bar_frame.copy()
for source in ("tradestats", "obstats", "orderstats"):
    feature_frame = resolve_feature_frame(source, feature_frames)
    
    # ИСПРАВЛЕНО: правильный порядок аргументов + исходные данные вместо validated
    attached = jfp._attach_source(causal, feature_frame, data[source], source)
    
    # Поддержка helper, возвращающего либо изменённый frame, либо None in-place.
    if isinstance(attached, pd.DataFrame):
        causal = attached
    print(f"{source:12s} attached → {causal.shape}")

# Добавляем base features по точному bar_ts.
base_cols = [c for c in base_raw.columns if c != "bar_ts" and pd.api.types.is_numeric_dtype(base_raw[c])]
if len(base_cols) != 73:
    raise RuntimeError(f"Ожидалось 73 base features, найдено {len(base_cols)}.")

base_join = base_raw[["bar_ts"] + base_cols].copy()
base_join = base_join.rename(columns={c: f"base__{c}" for c in base_cols})
base_cols = [c for c in base_join.columns if c.startswith("base__")]

if "bar_ts" not in causal.columns:
    raise RuntimeError(f"После causal join нет bar_ts. Колонки: {list(causal.columns)[:30]}")

causal["bar_ts"] = pd.to_datetime(causal["bar_ts"], errors="coerce", utc=True)
base_join["bar_ts"] = pd.to_datetime(base_join["bar_ts"], errors="coerce", utc=True)
causal = causal.merge(base_join, on="bar_ts", how="left", validate="one_to_one")

# Identify features and provenance without admitting timestamps into X.
time_cols = {"bar_ts", "bar_ts_utc", "ts", "ts_utc", "close_time"}
provenance_cols = [
    c for c in causal.columns
    if c.endswith("__event_ts") or c.endswith("__available_at")
]
sc_prefixes = ("tradestats__", "obstats__", "orderstats__")
sc_feature_cols = [
    c for c in causal.columns
    if c.startswith(sc_prefixes)
    and c not in provenance_cols
    and pd.api.types.is_numeric_dtype(causal[c])
]
# Не включаем сырые OHLCV/служебные колонки из bars: только канонические 73 + 401.
feature_cols = base_cols + sc_feature_cols

if len(feature_cols) != EXPECTED_FEATURES:
    raise RuntimeError(
        f"Feature inventory mismatch: {len(feature_cols)} numeric ML features, expected {EXPECTED_FEATURES}. "
        "Не продолжаем обучение на непроверенной матрице."
    )

if len(feature_cols) != len(set(feature_cols)):
    raise RuntimeError("Есть дублирующиеся имена признаков.")

FINAL_DATASET = causal[["bar_ts"] + feature_cols + provenance_cols].copy()
FINAL_DATASET = FINAL_DATASET.sort_values("bar_ts").reset_index(drop=True)
X_raw = FINAL_DATASET[feature_cols].copy()

print("FINAL_DATASET:", FINAL_DATASET.shape)
print("Rows:", len(FINAL_DATASET))
print("ML features:", len(feature_cols))
print("Provenance columns:", len(provenance_cols))

tradestats   attached → (8634, 106)
obstats      attached → (8634, 244)
orderstats   attached → (8634, 417)
FINAL_DATASET: (8634, 481)
Rows: 8634
ML features: 474
Provenance columns: 6


## 3. Temporal Audit — статус PASSED

По предыдущей диагностике пользователя статус Temporal Audit = `PASSED`. В этом исследовании не повторяем полный аудит доступности: дальше работаем над моделями и торговой проверкой. Оставляем только короткую проверку контракта provenance, чтобы notebook не продолжил работу, если обязательные поля неожиданно отсутствуют.

In [51]:
# Lightweight prerequisite check only; this does not rerun the already-passed temporal audit.
required_provenance = [
    f"{source}__available_at" for source in ("tradestats", "obstats", "orderstats")
] + [
    f"{source}__event_ts" for source in ("tradestats", "obstats", "orderstats")
]
missing_provenance = [col for col in required_provenance if col not in FINAL_DATASET.columns]
if missing_provenance:
    raise RuntimeError(f"Temporal Audit prerequisite missing provenance columns: {missing_provenance}")
TEMPORAL_AUDIT_STATUS = "PASSED (previously confirmed; full audit not rerun in this research notebook)"
print(TEMPORAL_AUDIT_STATUS)
print("Provenance columns present:", len(required_provenance))

PASSED (previously confirmed; full audit not rerun in this research notebook)
Provenance columns present: 6


## 4. Проверка качества признаков

Нулевые значения не удаляем автоматически. Важные показатели:
- `NaN` и `±inf` по каждому признаку;
- доля нулей (диагностика, не критерий удаления);
- константные признаки;
- полностью пустые признаки;
- дублирующиеся строки и timestamps.

Исходный `FINAL_DATASET` не перезаписывается. Для модели создаётся отдельная копия, в которой бесконечности переводятся в `NaN`; далее imputer обучается только на train.

In [52]:
X_num = X_raw.apply(pd.to_numeric, errors="coerce")
arr = X_num.to_numpy(dtype="float64", copy=True)
finite_mask = np.isfinite(arr)

quality = pd.DataFrame(index=X_num.columns)
quality["missing_pct"] = X_num.isna().mean() * 100
quality["positive_inf"] = np.isposinf(arr).sum(axis=0)
quality["negative_inf"] = np.isneginf(arr).sum(axis=0)
quality["zero_pct"] = (X_num.eq(0).mean() * 100)
quality["unique_values"] = X_num.nunique(dropna=True)
quality["all_missing"] = X_num.isna().all()
quality["constant_or_single_value"] = quality["unique_values"] <= 1
quality["non_finite_total"] = quality["positive_inf"] + quality["negative_inf"]

print("Rows:", len(X_num), "| features:", X_num.shape[1])
print("NaN cells:", int(X_num.isna().sum().sum()))
print("+inf cells:", int(np.isposinf(arr).sum()))
print("-inf cells:", int(np.isneginf(arr).sum()))
print("Zero cells (% of all feature cells):", round(float(X_num.eq(0).to_numpy().mean() * 100), 2))
print("Duplicate bar_ts:", int(FINAL_DATASET["bar_ts"].duplicated().sum()))
print("Duplicate feature rows:", int(X_num.duplicated().sum()))

print("\\nWorst missingness:")
display(quality.sort_values(["missing_pct", "non_finite_total"], ascending=False).head(30))

print("\\nFeatures with infinities:")
display(quality.loc[quality["non_finite_total"] > 0].sort_values("non_finite_total", ascending=False))

print("\\nConstant / all-missing features:")
display(quality.loc[quality["constant_or_single_value"] | quality["all_missing"]])

Rows: 8634 | features: 474
NaN cells: 31109
+inf cells: 0
-inf cells: 0
Zero cells (% of all feature cells): 0.58
Duplicate bar_ts: 0
Duplicate feature rows: 0
\nWorst missingness:


,missing_pct,positive_inf,negative_inf,zero_pct,unique_values,all_missing,constant_or_single_value,non_finite_total
tradestats__ts_flow_x_return_ret,6.265925,0,0,0.023164,2625,False,False,0
obstats__ob_vol_imb_bbo_vs_deep_ret,3.428307,0,0,0.880241,1893,False,False,0
tradestats__ts_vwap_b_s_bps_ret,3.254575,0,0,0.544360,2821,False,False,0
obstats__ob_val_imb_bbo_vs_deep_ret,1.714153,0,0,1.042391,1848,False,False,0
tradestats__ts_trade_imbalance_ret,1.204540,0,0,0.000000,2962,False,False,0
orderstats__os_cancel_orders_imb_ret,1.181376,0,0,0.000000,2991,False,False,0
orderstats__os_put_orders_imb_ret,1.007644,0,0,0.000000,2999,False,False,0
obstats__ob_total_volume_z3,0.926569,0,0,0.000000,2944,False,False,0
obstats__ob_total_volume_ma3,0.926569,0,0,0.000000,2944,False,False,0
obstats__ob_total_volume_z6,0.926569,0,0,0.000000,2944,False,False,0


\nFeatures with infinities:


,missing_pct,positive_inf,negative_inf,zero_pct,unique_values,all_missing,constant_or_single_value,non_finite_total


\nConstant / all-missing features:


,missing_pct,positive_inf,negative_inf,zero_pct,unique_values,all_missing,constant_or_single_value,non_finite_total


In [53]:
# Separate ML copy: keep source dataset unchanged.
X_model = X_num.replace([np.inf, -np.inf], np.nan).copy()

# Remove only features that cannot carry information at all (all missing or constant).
drop_features = quality.index[quality["all_missing"] | quality["constant_or_single_value"]].tolist()
if drop_features:
    print("Removing non-informative columns from model copy:", len(drop_features))
    print(drop_features[:30])
    X_model = X_model.drop(columns=drop_features)

print("X_model shape:", X_model.shape)
print("Remaining NaN cells (to be handled by train-only imputer):", int(X_model.isna().sum().sum()))

# Optional strict stop for pathological data; does not delete rows.
if X_model.shape[1] == 0:
    raise RuntimeError("После удаления пустых/константных признаков не осталось признаков.")

X_model shape: (8634, 474)
Remaining NaN cells (to be handled by train-only imputer): 31109


## 2. Фиксируем признаки и строим labels внутри каждой конфигурации

`FINAL_DATASET` и `X_model` построены на одних и тех же Dollar Bars. На каждом trial заново строятся CUSUM events и Triple Barrier labels; затем признаки присоединяются по timestamp. `TIME` исключается из бинарной цели, как и в предыдущей версии.

Важная методологическая оговорка: бэктест входит на следующем баре, тогда как Triple Barrier рассчитывает уровни от цены события. В симуляции ниже расстояния TP/SL переносятся на фактический entry. Это воспроизводит прежнюю логику, но gap между сигналом и входом остаётся приближением. После первого прогона стоит отдельно сравнить с исполнением, где уровни пересчитываются строго от entry.

In [54]:
# Configuration and imports for the optimization stage
import time, warnings
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, brier_score_loss
from src.data.labeling.cusum import cusum_filter
from src.data.labeling.triple_barrier import triple_barrier_labeling

try:
    import optuna
except ImportError as exc:
    raise ImportError(
        "Для этого notebook нужен optuna. Установи в активном окружении: pip install optuna"
    ) from exc

SEED = 63
INITIAL_CAPITAL = 100_000.0
COMMISSION = 0.0006   # за каждую сторону сделки
SLIPPAGE = 0.0001     # adverse slippage на каждой стороне
DOLLAR_THRESHOLD_FIXED = DOLLAR_THRESHOLD
DOLLAR_THRESHOLD_CANDIDATES = [5_000_000, 7_500_000, 10_000_000, 15_000_000]
TARGET_WINDOW_FIXED = TARGET_WINDOW if "TARGET_WINDOW" in globals() else 55

# Небольшой бюджет по умолчанию; сначала проверь корректность на 20–30 trials.
N_TRIALS_PER_OUTER_FOLD = 5  # 75 trials total; feature rebuild is cached per threshold
MIN_TRADES_PER_OPTIMIZATION = 8
RF_N_ESTIMATORS = 350
RF_MAX_DEPTH = 8
RF_MIN_SAMPLES_LEAF = 8
RF_MAX_FEATURES = "sqrt"

# Outer validation periods are chronological validation windows, not train intervals.
# Train always contains all earlier history, after purge.
OUTER_FOLDS = [
    ("WF1", 0.50, 0.65),
    ("WF2", 0.65, 0.80),
    ("WF3", 0.80, 1.00),
]

print("Ticker:", TICKER)
print("Dollar threshold fixed for this experiment:", DOLLAR_THRESHOLD_FIXED)
print("RF is fixed; Optuna tunes data/label/trading configuration only.")
print("Optuna version:", optuna.__version__)


Ticker: VTBR
Dollar threshold fixed for this experiment: 10000000
RF is fixed; Optuna tunes data/label/trading configuration only.
Optuna version: 4.6.0


In [55]:
# Build an immutable bar-level feature table. Do not align it to one fixed set of labels.
if "bar_ts" not in FINAL_DATASET.columns:
    raise RuntimeError("FINAL_DATASET must contain bar_ts.")
feature_cols = list(X_model.columns)
feature_table = pd.concat(
    [pd.to_datetime(FINAL_DATASET["bar_ts"], errors="coerce", utc=True).rename("bar_ts").reset_index(drop=True),
     X_model.reset_index(drop=True)],
    axis=1
)
feature_table = (
    feature_table.dropna(subset=["bar_ts"])
    .sort_values("bar_ts")
    .drop_duplicates("bar_ts", keep="last")
    .set_index("bar_ts")
)
feature_table = feature_table[feature_cols].replace([np.inf, -np.inf], np.nan)

bars_bt = bars.copy()
bars_bt["bar_ts"] = pd.to_datetime(bars_bt["bar_ts"], errors="coerce", utc=True)
for low, up in [("open","OPEN"), ("high","HIGH"), ("low","LOW"), ("close","CLOSE")]:
    if low not in bars_bt.columns and up in bars_bt.columns:
        bars_bt[low] = bars_bt[up]
    bars_bt[low] = pd.to_numeric(bars_bt[low], errors="coerce")
bars_bt = (
    bars_bt.dropna(subset=["bar_ts","open","high","low","close"])
    .sort_values("bar_ts")
    .drop_duplicates("bar_ts", keep="last")
    .reset_index(drop=True)
)
bar_ts_index = pd.DatetimeIndex(bars_bt["bar_ts"])
if not feature_table.index.is_monotonic_increasing:
    raise RuntimeError("Feature table timestamps are not sorted.")
if feature_table.index.has_duplicates:
    raise RuntimeError("Feature table contains duplicate timestamps.")

print("Bars:", len(bars_bt), "| feature rows:", len(feature_table), "| features:", len(feature_cols))
print("Period:", bar_ts_index.min(), "→", bar_ts_index.max())
display(pd.DataFrame([{
    "ticker": TICKER, "bars": len(bars_bt), "features": len(feature_cols),
    "start": str(bar_ts_index.min()), "end": str(bar_ts_index.max()),
    "dollar_threshold_fixed": DOLLAR_THRESHOLD_FIXED
}]))


# Freeze calendar boundaries from the baseline feature timeline. Changing Dollar Bars
# changes event timestamps/count, but must not move the external OOS calendar periods.
BASE_TIMELINE = pd.DatetimeIndex(feature_table.index).sort_values().unique()


Bars: 8634 | feature rows: 8634 | features: 474
Period: 2026-08-28 04:02:32+00:00 → 2026-10-15 20:37:59+00:00


,ticker,bars,features,start,end,dollar_threshold_fixed
0,VTBR,8634,474,2026-08-28 04:02:32+00:00,2026-10-15 20:37:59+00:00,10000000


In [56]:
def build_feature_context_for_threshold(threshold):
    """Rebuild Dollar Bars and the full causal feature matrix for a given threshold.

    This intentionally reruns the expensive feature pipeline. The caller caches the
    result by threshold, so each candidate threshold is rebuilt at most once per run.
    """
    # Dollar Bars: используем тот же денежный объём VALUE / PRICE, что и в reference.
    bars = jfp.build_dollar_bars_from_trades(data["trades"], threshold=threshold)
    if bars is None or bars.empty:
        raise RuntimeError("Dollar Bar builder вернул пустой результат.")

    # Унифицируем имя времени для последующих causal joins.
    if "bar_ts" not in bars.columns:
        if "close_time" not in bars.columns:
            raise RuntimeError(f"В bars нет bar_ts/close_time. Колонки: {list(bars.columns)}")
        bars["bar_ts"] = pd.to_datetime(bars["close_time"], errors="coerce", utc=True)
    else:
        bars["bar_ts"] = pd.to_datetime(bars["bar_ts"], errors="coerce", utc=True)

    bars = (bars.dropna(subset=["bar_ts"])
                 .sort_values("bar_ts")
                 .drop_duplicates("bar_ts", keep="last")
                 .reset_index(drop=True))


    # Base 73 + Super Candle 401. Работаем через helper-функции pipeline.
    base_raw = jfp._base(bars.copy())
    if isinstance(base_raw, tuple):
        # Поддержка версий helper, возвращающих (features, metadata).
        base_raw = base_raw[0]
    if not isinstance(base_raw, pd.DataFrame):
        raise TypeError(f"_base должен вернуть DataFrame, получено: {type(base_raw)}")

    # Приводим base frame к контракту bar_ts + 73 числовых признака.
    if "bar_ts" not in base_raw.columns:
        if "close_time" in base_raw.columns:
            base_raw = base_raw.rename(columns={"close_time": "bar_ts"})
        elif isinstance(base_raw.index, pd.DatetimeIndex):
            base_raw = base_raw.reset_index().rename(columns={base_raw.index.name or "index": "bar_ts"})
        elif "ts" in base_raw.columns:
            base_raw = base_raw.rename(columns={"ts": "bar_ts"})
        else:
            raise RuntimeError(f"Не найден временной ключ в _base output. Колонки: {list(base_raw.columns)[:25]}")

    base_raw["bar_ts"] = pd.to_datetime(base_raw["bar_ts"], errors="coerce", utc=True)
    base_raw = base_raw.dropna(subset=["bar_ts"]).sort_values("bar_ts").drop_duplicates("bar_ts", keep="last").reset_index(drop=True)

    # _build_sc_features может возвращать feature_frames либо (feature_frames, groups).
    # Адаптируем Dollar Bars к формату, ожидаемому pipeline.
    bars_for_sc = bars.copy()

    # В pipeline ожидаются ts и CLOSE.
    if "ts" not in bars_for_sc.columns:
        if "bar_ts" in bars_for_sc.columns:
            bars_for_sc["ts"] = bars_for_sc["bar_ts"]
        elif "close_time" in bars_for_sc.columns:
            bars_for_sc["ts"] = bars_for_sc["close_time"]
        else:
            raise KeyError("В bars отсутствует временная колонка ts/bar_ts/close_time")

    if "CLOSE" not in bars_for_sc.columns:
        if "close" in bars_for_sc.columns:
            bars_for_sc["CLOSE"] = bars_for_sc["close"]
        else:
            raise KeyError("В bars отсутствует цена закрытия CLOSE/close")

    bars_for_sc["ts"] = pd.to_datetime(bars_for_sc["ts"], errors="coerce", utc=True)

    bars_for_sc = (
        bars_for_sc.dropna(subset=["ts"])
        .sort_values("ts")
        .reset_index(drop=True))

    sc_result = jfp._build_sc_features(data, bars_for_sc)

    if isinstance(sc_result, tuple):
        feature_frames = sc_result[0]
        feature_groups = sc_result[1] if len(sc_result) > 1 else {}
    else:
        feature_frames = sc_result
        feature_groups = {}

    if not isinstance(feature_frames, dict):
        raise TypeError("_build_sc_features должен возвращать dict feature frames.")


    # Availability validation and causal joins, source by source.
    # Helpers' outputs are inspected to support the current prepared module contract.
    validated = {}
    availability_diagnostics = []

    for source in ("tradestats", "obstats", "orderstats"):
        if data[source].empty:
            raise RuntimeError(f"Локальный источник {source} пуст.")

        result = jfp._choose_availability(data[source].copy(), source)
        if isinstance(result, tuple):
            validated[source] = result[0]
            if len(result) > 1 and isinstance(result[1], pd.DataFrame):
                availability_diagnostics.append(result[1])
        else:
            validated[source] = result

    # Normalize the bars frame for the pipeline's attach helper.
    bar_frame = bars.copy()
    if "bar_ts" not in bar_frame.columns:
        raise RuntimeError("bar_ts missing from Dollar Bars.")

    # Resolve each raw source to the corresponding feature frame by name.
    def resolve_feature_frame(source, frames):
        if source in frames:
            return frames[source]
        matches = [k for k in frames if source.lower() in str(k).lower()]
        if len(matches) == 1:
            return frames[matches[0]]
        raise RuntimeError(f"Не удалось однозначно сопоставить {source} с feature frame. Доступны: {list(frames)}")

    causal = bar_frame.copy()
    for source in ("tradestats", "obstats", "orderstats"):
        feature_frame = resolve_feature_frame(source, feature_frames)
    
        # ИСПРАВЛЕНО: правильный порядок аргументов + исходные данные вместо validated
        attached = jfp._attach_source(causal, feature_frame, data[source], source)
    
        # Поддержка helper, возвращающего либо изменённый frame, либо None in-place.
        if isinstance(attached, pd.DataFrame):
            causal = attached

    # Добавляем base features по точному bar_ts.
    base_cols = [c for c in base_raw.columns if c != "bar_ts" and pd.api.types.is_numeric_dtype(base_raw[c])]
    if len(base_cols) != 73:
        raise RuntimeError(f"Ожидалось 73 base features, найдено {len(base_cols)}.")

    base_join = base_raw[["bar_ts"] + base_cols].copy()
    base_join = base_join.rename(columns={c: f"base__{c}" for c in base_cols})
    base_cols = [c for c in base_join.columns if c.startswith("base__")]

    if "bar_ts" not in causal.columns:
        raise RuntimeError(f"После causal join нет bar_ts. Колонки: {list(causal.columns)[:30]}")

    causal["bar_ts"] = pd.to_datetime(causal["bar_ts"], errors="coerce", utc=True)
    base_join["bar_ts"] = pd.to_datetime(base_join["bar_ts"], errors="coerce", utc=True)
    causal = causal.merge(base_join, on="bar_ts", how="left", validate="one_to_one")

    # Identify features and provenance without admitting timestamps into X.
    time_cols = {"bar_ts", "bar_ts_utc", "ts", "ts_utc", "close_time"}
    provenance_cols = [
        c for c in causal.columns
        if c.endswith("__event_ts") or c.endswith("__available_at")
    ]
    sc_prefixes = ("tradestats__", "obstats__", "orderstats__")
    sc_feature_cols = [
        c for c in causal.columns
        if c.startswith(sc_prefixes)
        and c not in provenance_cols
        and pd.api.types.is_numeric_dtype(causal[c])
    ]
    # Не включаем сырые OHLCV/служебные колонки из bars: только канонические 73 + 401.
    feature_cols = base_cols + sc_feature_cols

    if len(feature_cols) != EXPECTED_FEATURES:
        raise RuntimeError(
            f"Feature inventory mismatch: {len(feature_cols)} numeric ML features, expected {EXPECTED_FEATURES}. "
            "Не продолжаем обучение на непроверенной матрице."
        )

    if len(feature_cols) != len(set(feature_cols)):
        raise RuntimeError("Есть дублирующиеся имена признаков.")

    FINAL_DATASET = causal[["bar_ts"] + feature_cols + provenance_cols].copy()
    FINAL_DATASET = FINAL_DATASET.sort_values("bar_ts").reset_index(drop=True)
    X_raw = FINAL_DATASET[feature_cols].copy()


    return bars, FINAL_DATASET, feature_cols

FEATURE_CONTEXT_CACHE = {}

def activate_dollar_threshold(threshold):
    """Switch all globals used by labels/backtest to a fully rebuilt threshold context."""
    global bars_bt, feature_table, bar_ts_index, DOLLAR_THRESHOLD_ACTIVE
    key = int(threshold)
    if key not in FEATURE_CONTEXT_CACHE:
        rebuilt_bars, rebuilt_dataset, rebuilt_cols = build_feature_context_for_threshold(key)
        ds = rebuilt_dataset.copy()
        ds["bar_ts"] = pd.to_datetime(ds["bar_ts"], errors="coerce", utc=True)
        xx = ds[rebuilt_cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
        ft = pd.concat([ds["bar_ts"].reset_index(drop=True), xx.reset_index(drop=True)], axis=1)
        ft = ft.dropna(subset=["bar_ts"]).sort_values("bar_ts").drop_duplicates("bar_ts", keep="last")
        ft = ft.set_index("bar_ts")
        bb = rebuilt_bars.copy()
        if "bar_ts" not in bb.columns:
            bb["bar_ts"] = pd.to_datetime(bb["close_time"], errors="coerce", utc=True)
        else:
            bb["bar_ts"] = pd.to_datetime(bb["bar_ts"], errors="coerce", utc=True)
        for low, up in [("open","OPEN"), ("high","HIGH"), ("low","LOW"), ("close","CLOSE")]:
            if low not in bb.columns and up in bb.columns:
                bb[low] = bb[up]
            bb[low] = pd.to_numeric(bb[low], errors="coerce")
        bb = bb.dropna(subset=["bar_ts","open","high","low","close"]).sort_values("bar_ts").drop_duplicates("bar_ts", keep="last").reset_index(drop=True)
        FEATURE_CONTEXT_CACHE[key] = (bb, ft)
    bars_bt, feature_table = FEATURE_CONTEXT_CACHE[key]
    bar_ts_index = pd.DatetimeIndex(bars_bt["bar_ts"])
    DOLLAR_THRESHOLD_ACTIVE = key
    return len(bars_bt), len(feature_table)


In [57]:
def build_trial_dataset(cusum_threshold, tp_mult, sl_mult, vertical_horizon, target_window):
    """Rebuild CUSUM + Triple Barrier labels for one trial; features remain bar-aligned."""
    b = bars_bt.set_index("bar_ts", drop=False).copy()
    b.index = pd.DatetimeIndex(b.index)
    b.index.name = "bar_ts"
    close = b["close"].astype(float)
    events = pd.DatetimeIndex(cusum_filter(close, threshold=float(cusum_threshold)))
    events = events[events.isin(b.index)].drop_duplicates().sort_values()
    if len(events) < 100:
        return None

    label_bars = b.rename(columns={
        "open":"OPEN", "high":"HIGH", "low":"LOW", "close":"CLOSE"
    })
    label_bars = label_bars[["OPEN","HIGH","LOW","CLOSE"]].copy()
    label_bars.index = pd.DatetimeIndex(label_bars.index)
    labels = triple_barrier_labeling(
        bars=label_bars,
        events=events,
        pt_sl=(float(tp_mult), float(sl_mult)),
        vertical_barrier=int(vertical_horizon),
        target_window=int(target_window),
        target_method="ewm",
    )
    if labels is None or labels.empty or not {"label","t1"}.issubset(labels.columns):
        return None
    labels = labels.copy()
    labels.index = pd.to_datetime(labels.index, errors="coerce", utc=True)
    labels = labels[~labels.index.isna()].sort_index()
    labels = labels[labels["label"].isin([-1,0,1])].copy()
    labels["label"] = labels["label"].astype(int)
    labels["t1"] = pd.to_datetime(labels["t1"], errors="coerce", utc=True)
    common = feature_table.index.intersection(labels.index)
    labels = labels.loc[common].sort_index()
    X = feature_table.reindex(labels.index).copy()
    mask = labels["label"].isin([-1,1]) & labels["t1"].notna()
    X = X.loc[mask]
    labels = labels.loc[mask].copy()
    y = labels["label"].map({-1:0, 1:1}).astype(int)
    if len(X) < 150 or y.nunique() < 2:
        return None
    return X, y, labels


def make_rf():
    # Fixed model; these are deliberately NOT Optuna parameters.
    return RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS,
        max_depth=RF_MAX_DEPTH,
        min_samples_leaf=RF_MIN_SAMPLES_LEAF,
        max_features=RF_MAX_FEATURES,
        class_weight="balanced_subsample",
        random_state=SEED,
        n_jobs=-1,
    )


def purge_train_indices(train_idx, labels, validation_start):
    # Purge events whose label interval reaches or crosses the validation start.
    t1 = labels.reindex(train_idx)["t1"]
    return train_idx[(t1.notna() & (t1 < validation_start)).to_numpy()]


def temporal_folds(index, specs):
    """Specs are validation start/end fractions; training is all earlier history."""
    n = len(index)
    folds = []
    for name, val_start_frac, val_end_frac in specs:
        a = max(1, int(n * val_start_frac))
        b = min(n, int(n * val_end_frac))
        if b <= a:
            continue
        tr0 = index[:a]
        va = index[a:b]
        if len(tr0) < 80 or len(va) < 20:
            continue
        folds.append({"name":name, "train_candidate":tr0, "val_idx":va,
                      "val_start":va[0], "val_end":index[b-1]})
    return folds


def prepare_xy(X_train, y_train, X_valid):
    # Fit imputer only on training data; no validation leakage.
    imp = SimpleImputer(strategy="median")
    a = pd.DataFrame(imp.fit_transform(X_train), index=X_train.index, columns=X_train.columns)
    b = pd.DataFrame(imp.transform(X_valid), index=X_valid.index, columns=X_valid.columns)
    # RF cannot use all-NaN columns; imputer may preserve them as zeros/NaNs depending on sklearn version.
    a = a.replace([np.inf,-np.inf], np.nan).fillna(0.0)
    b = b.replace([np.inf,-np.inf], np.nan).fillna(0.0)
    return a, b


## 3. Execution simulator and objective

Чистая доходность оценивается после комиссии и проскальзывания. Позиция закрывается по TP/SL, по противоположному уверенно прогнозируемому сигналу (исполнение на следующем open) или по vertical horizon. Если в одном OHLC-баре коснулись TP и SL, предполагается SL первым.

Фактический TP% не равен `tp_mult`: multiplier применяется к динамическому target volatility. Поэтому в `oos_trades` теперь есть `tp_distance_pct` и `sl_distance_pct`. Если бизнес-правило требует TP не меньше 0.6%, это надо задавать отдельным ограничением на фактическую дистанцию, а не трактовать `tp_mult=0.6` как 0.6%.


In [58]:
def simulate_oos_fold(fold, probs, labels, vertical_horizon, confidence_threshold):
    """
    One-position-at-a-time OHLC simulator.
    Entry: next bar open after a signal at close.
    Exit: TP/SL, reverse confident signal at next bar open, or vertical horizon.
    If TP and SL are both touched in one bar, conservatively assume SL first.
    """
    probs = probs.dropna().sort_index()
    empty = {"total_return":0.0, "trades":0, "win_rate":np.nan,
             "profit_factor":0.0, "max_drawdown":0.0, "trade_rows":[]}
    if probs.empty:
        return empty

    confidence = np.maximum(probs.to_numpy(), 1.0 - probs.to_numpy())
    side_arr = np.where(probs.to_numpy() >= 0.5, 1, -1)
    side_arr[confidence < float(confidence_threshold)] = 0
    signals = pd.Series(side_arr, index=probs.index)
    signal_by_ts = signals.to_dict()
    ts_to_pos = {ts:i for i,ts in enumerate(bar_ts_index)}

    equity = float(INITIAL_CAPITAL)
    peak = equity
    max_dd = 0.0
    last_exit = -1
    trade_rows = []
    fold_end = fold["val_end"]

    for signal_ts, side in signals.items():
        if side == 0 or signal_ts not in ts_to_pos:
            continue
        sig_pos = ts_to_pos[signal_ts]
        entry_pos = sig_pos + 1
        if entry_pos >= len(bars_bt) or entry_pos <= last_exit:
            continue
        if bars_bt.iloc[entry_pos]["bar_ts"] > fold_end:
            continue

        row = labels.loc[signal_ts]
        signal_close = float(bars_bt.iloc[sig_pos]["close"])
        try:
            upper, lower = float(row.get("upper", np.nan)), float(row.get("lower", np.nan))
        except Exception:
            continue
        if not (np.isfinite(upper) and np.isfinite(lower) and upper > signal_close > lower):
            continue

        if side == 1:
            tp_dist, sl_dist = upper - signal_close, signal_close - lower
        else:
            tp_dist, sl_dist = signal_close - lower, upper - signal_close
        if tp_dist <= 0 or sl_dist <= 0:
            continue

        raw_entry = float(bars_bt.iloc[entry_pos]["open"])
        entry_px = raw_entry * (1 + SLIPPAGE if side == 1 else 1 - SLIPPAGE)
        tp_px = entry_px + tp_dist if side == 1 else entry_px - tp_dist
        sl_px = entry_px - sl_dist if side == 1 else entry_px + sl_dist
        max_exit_pos = min(len(bars_bt)-1, entry_pos + int(vertical_horizon)-1)
        if bars_bt.iloc[max_exit_pos]["bar_ts"] > fold_end:
            continue

        exit_pos, reason = max_exit_pos, "VERTICAL_CLOSE"
        raw_exit = float(bars_bt.iloc[max_exit_pos]["close"])
        for j in range(entry_pos, max_exit_pos+1):
            hi, lo = float(bars_bt.iloc[j]["high"]), float(bars_bt.iloc[j]["low"])
            hit_sl = (lo <= sl_px) if side == 1 else (hi >= sl_px)
            hit_tp = (hi >= tp_px) if side == 1 else (lo <= tp_px)
            if hit_sl:
                exit_pos, reason, raw_exit = j, "SL", sl_px
                break
            if hit_tp:
                exit_pos, reason, raw_exit = j, "TP", tp_px
                break

            # Reverse signal is only known after bar j closes; fill at next bar open.
            ts_j = bars_bt.iloc[j]["bar_ts"]
            reverse_side = signal_by_ts.get(ts_j, 0)
            if reverse_side == -side and j + 1 < len(bars_bt):
                if bars_bt.iloc[j+1]["bar_ts"] <= fold_end:
                    exit_pos, reason = j+1, "REVERSE_SIGNAL"
                    raw_exit = float(bars_bt.iloc[j+1]["open"])
                    break

        exit_px = raw_exit * (1 - SLIPPAGE if side == 1 else 1 + SLIPPAGE)
        qty = equity / entry_px if entry_px > 0 else 0.0
        gross_pnl = qty * (exit_px-entry_px) * side
        fees = COMMISSION * (abs(qty*entry_px) + abs(qty*exit_px))
        net_pnl = gross_pnl - fees
        start_equity = equity
        equity += net_pnl
        peak = max(peak, equity)
        max_dd = max(max_dd, (peak-equity)/peak if peak > 0 else 0.0)

        trade_rows.append({
            "signal_ts":signal_ts, "entry_ts":bars_bt.iloc[entry_pos]["bar_ts"],
            "exit_ts":bars_bt.iloc[exit_pos]["bar_ts"], "side":"LONG" if side==1 else "SHORT",
            "p_up":float(probs.loc[signal_ts]), "confidence":float(max(probs.loc[signal_ts],1-probs.loc[signal_ts])),
            "entry_px":entry_px, "exit_px":exit_px, "exit_reason":reason,
            "tp_distance_pct":tp_dist / signal_close,
            "sl_distance_pct":sl_dist / signal_close,
            "gross_pnl":gross_pnl, "fees":fees, "net_pnl":net_pnl,
            "trade_return":net_pnl/start_equity if start_equity else np.nan,
            "equity_after":equity,
        })
        last_exit = exit_pos

    wins = [r["net_pnl"] for r in trade_rows if r["net_pnl"] > 0]
    losses = [-r["net_pnl"] for r in trade_rows if r["net_pnl"] < 0]
    pf = (sum(wins)/sum(losses)) if sum(losses) > 0 else (float("inf") if wins else 0.0)
    return {
        "total_return": equity/INITIAL_CAPITAL - 1.0,
        "trades":len(trade_rows),
        "win_rate":float(np.mean([r["net_pnl"]>0 for r in trade_rows])) if trade_rows else np.nan,
        "profit_factor":float(pf), "max_drawdown":float(max_dd), "trade_rows":trade_rows,
    }


def eval_model_fold(fold, X, y, labels, vertical_horizon, confidence_threshold):
    tr_candidate = fold["train_candidate"]
    tr_idx = purge_train_indices(tr_candidate, labels, fold["val_start"])
    va_idx = fold["val_idx"]
    if len(tr_idx) < 80 or len(va_idx) < 20 or y.loc[tr_idx].nunique() < 2 or y.loc[va_idx].nunique() < 2:
        return None
    Xtr, Xva = X.loc[tr_idx], X.loc[va_idx]
    Xtr, Xva = prepare_xy(Xtr, y.loc[tr_idx], Xva)
    model = make_rf()
    t1 = labels.reindex(tr_idx)["t1"]
    duration = (t1 - pd.Series(tr_idx, index=tr_idx)).dt.total_seconds().clip(lower=1.0)
    weights = (1.0 / duration)
    weights = weights / weights.mean()
    model.fit(Xtr, y.loc[tr_idx], sample_weight=weights.to_numpy())
    p = model.predict_proba(Xva)[:,1]
    probs = pd.Series(p, index=va_idx)
    bt = simulate_oos_fold(fold, probs, labels, vertical_horizon, confidence_threshold)
    yv = y.loc[va_idx]
    try:
        auc = roc_auc_score(yv, p) if yv.nunique() == 2 else np.nan
    except Exception:
        auc = np.nan
    bt.update({"n_validation":len(va_idx), "balanced_accuracy":balanced_accuracy_score(yv, (p>=0.5).astype(int)),
               "roc_auc":auc, "brier":brier_score_loss(yv,p)})
    return bt


## 4. Nested walk-forward optimization

Каждый trial теперь выбирает `dollar_threshold` из дискретного списка и активирует полностью пересобранные Dollar Bars/признаки для этого порога. Это значительно дороже прежнего notebook; один и тот же threshold кэшируется, поэтому дорогостоящий feature build выполняется не на каждом trial, а максимум один раз на каждый уникальный кандидат за запуск.

Фолды — это календарные интервалы, зафиксированные по baseline timeline. Для каждого внешнего fold Optuna выбирает конфигурацию на внутреннем прошлом; затем выбранная конфигурация проверяется на внешнем периоде. Поэтому лучший внутренний `trial.value` может быть положительным, а внешняя доходность отрицательной — это не противоречие, а возможный признак переобучения.


In [59]:
def inner_specs_for_outer_train(n_events):
    # Three expanding validation windows within the outer training history.
    # Fractions are relative to the outer-train event set.
    specs = [("I1",0.40,0.55), ("I2",0.55,0.70), ("I3",0.70,0.85)]
    return specs


def make_outer_fold_specs():
    # Define boundaries from the full bar timeline, independent of trial labels.
    # This prevents changing CUSUM/TP/SL from changing which calendar period is OOS.
    timeline = BASE_TIMELINE
    n = len(timeline)
    out = []
    for name, a, b in OUTER_FOLDS:
        train_end_pos = max(1, int(n*a))
        val_end_pos = min(n, int(n*b))
        if val_end_pos <= train_end_pos:
            continue
        out.append({
            "name":name,
            "train_end_ts":timeline[train_end_pos-1],
            "val_start":timeline[train_end_pos],
            "val_end":timeline[val_end_pos-1],
        })
    return out

def make_inner_folds(train_index, labels):
    n = len(train_index)
    out = []
    for name, a, b in inner_specs_for_outer_train(n):
        i, j = int(n*a), min(n, int(n*b))
        if i < 80 or j-i < 20:
            continue
        val_end = train_index[j-1]
        val_idx = train_index[i:j]
        resolved = labels.reindex(val_idx)["t1"] <= val_end
        val_idx = val_idx[resolved.fillna(False).to_numpy()]
        if len(val_idx) < 20:
            continue
        out.append({
            "name":name, "train_candidate":train_index[:i],
            "val_idx":val_idx, "val_start":train_index[i],
            "val_end":val_end,
        })
    return out


def objective_for_outer_train(trial, outer_train_end_ts):
    dollar_threshold = trial.suggest_categorical("dollar_threshold", DOLLAR_THRESHOLD_CANDIDATES)
    n_bars, n_features = activate_dollar_threshold(dollar_threshold)
    # Avoid optimizing configurations with too little data for a stable fit.
    if n_bars < 300 or n_features < 150:
        return -1.0
    cusum_threshold = trial.suggest_float("cusum_threshold", 0.0003, 0.0030, log=True)
    tp_mult = trial.suggest_float("tp_mult", 0.30, 2.00, step=0.10)
    sl_mult = trial.suggest_float("sl_mult", 0.30, 2.00, step=0.10)
    vertical_horizon = trial.suggest_int("vertical_horizon", 10, 100, step=10)
    target_window = trial.suggest_int("target_window", 20, 100, step=10)
    confidence_threshold = trial.suggest_float("confidence_threshold", 0.50, 0.80, step=0.025)

    built = build_trial_dataset(cusum_threshold, tp_mult, sl_mult, vertical_horizon, target_window)
    if built is None:
        return -1.0
    X, y, labels = built
    # Keep only events inside the outer training calendar period.
    X = X.loc[X.index <= outer_train_end_ts]
    y = y.reindex(X.index)
    labels = labels.reindex(X.index)
    # Exclude training-period events whose outcomes are not fully known by outer cutoff.
    resolved_train = labels["t1"] < outer_train_end_ts
    X, y, labels = X.loc[resolved_train.fillna(False)], y.loc[resolved_train.fillna(False)], labels.loc[resolved_train.fillna(False)]
    if len(X) < 150 or y.nunique() < 2:
        return -1.0

    inner_folds = make_inner_folds(X.index, labels)
    if len(inner_folds) < 1:
        return -1.0
    fold_returns, fold_trades = [], []
    for f in inner_folds:
        result = eval_model_fold(f, X, y, labels, vertical_horizon, confidence_threshold)
        if result is None:
            continue
        fold_returns.append(result["total_return"])
        fold_trades.append(result["trades"])
    if not fold_returns:
        return -1.0

    # PnL-first objective: compounded net return, with a penalty for too few trades.
    compounded = float(np.prod(1.0 + np.asarray(fold_returns)) - 1.0)
    total_trades = int(np.sum(fold_trades))
    if total_trades < MIN_TRADES_PER_OPTIMIZATION:
        compounded -= 0.05 * (MIN_TRADES_PER_OPTIMIZATION-total_trades)/MIN_TRADES_PER_OPTIMIZATION
    # Penalize extreme fold instability without replacing PnL with a classifier metric.
    worst_fold = float(np.min(fold_returns))
    objective = compounded + 0.20 * worst_fold
    trial.set_user_attr("dollar_threshold", int(dollar_threshold))
    trial.set_user_attr("inner_fold_returns", [float(x) for x in fold_returns])
    trial.set_user_attr("inner_fold_trades", [int(x) for x in fold_trades])
    trial.set_user_attr("inner_compounded_return", compounded)
    return float(objective)


def run_nested_optuna(n_trials=N_TRIALS_PER_OUTER_FOLD):
    activate_dollar_threshold(DOLLAR_THRESHOLD_FIXED)
    # The event index used to define outer calendar slices comes from the baseline labels.
    baseline = build_trial_dataset(CUSUM_THRESHOLD, PT_SL[0], PT_SL[1], VERTICAL_HORIZON, TARGET_WINDOW_FIXED)
    if baseline is None:
        raise RuntimeError("Baseline labels are empty; cannot define outer walk-forward periods.")
    X0, y0, labels0 = baseline
    outer_folds = make_outer_fold_specs()
    all_results, best_params_by_fold, all_trials = [], {}, []

    for outer in outer_folds:
        print(f"\n=== {outer['name']} | train ends={outer['train_end_ts']} | validation={outer['val_start']} → {outer['val_end']} ===")
        study = optuna.create_study(
            direction="maximize",
            sampler=optuna.samplers.TPESampler(seed=SEED),
            pruner=optuna.pruners.NopPruner(),
        )
        study.optimize(lambda trial: objective_for_outer_train(trial, outer["train_end_ts"]),
                       n_trials=int(n_trials), show_progress_bar=True, gc_after_trial=True)
        params = dict(study.best_params)
        best_params_by_fold[outer["name"]] = params
        trial_df = study.trials_dataframe(attrs=("number","value","params","state"))
        trial_df["outer_fold"] = outer["name"]
        all_trials.append(trial_df)

        activate_dollar_threshold(params["dollar_threshold"])
        built = build_trial_dataset(
            params["cusum_threshold"], params["tp_mult"], params["sl_mult"],
            params["vertical_horizon"], params["target_window"]
        )
        if built is None:
            all_results.append({"fold":outer["name"], **params, "total_return":np.nan,
                                "trades":0, "status":"NO_DATA"})
            continue
        X, y, labels = built
        tr_idx = X.index[X.index <= outer["train_end_ts"]]
        va_idx = X.index[(X.index >= outer["val_start"]) & (X.index <= outer["val_end"])]
        va_idx = va_idx[(labels.reindex(va_idx)["t1"] <= outer["val_end"]).fillna(False).to_numpy()]
        if len(tr_idx) < 80 or len(va_idx) < 20:
            all_results.append({"fold":outer["name"], **params, "total_return":np.nan,
                                "trades":0, "status":"TOO_FEW_EVENTS"})
            continue
        f = {"name":outer["name"], "train_candidate":tr_idx, "val_idx":va_idx,
             "val_start":outer["val_start"], "val_end":outer["val_end"]}
        result = eval_model_fold(f, X, y, labels, params["vertical_horizon"], params["confidence_threshold"])
        if result is None:
            all_results.append({"fold":outer["name"], **params, "total_return":np.nan,
                                "trades":0, "status":"FOLD_SKIPPED"})
            continue
        all_results.append({
            "fold":outer["name"], **params,
            "total_return":result["total_return"], "trades":result["trades"],
            "win_rate":result["win_rate"], "profit_factor":result["profit_factor"],
            "max_drawdown":result["max_drawdown"], "balanced_accuracy":result["balanced_accuracy"],
            "roc_auc":result["roc_auc"], "brier":result["brier"], "status":"OK",
        })

    return pd.DataFrame(all_results), best_params_by_fold, pd.concat(all_trials, ignore_index=True)


outer_results, best_params_by_fold, optuna_trials = run_nested_optuna()
display(outer_results)
display(pd.DataFrame([{"outer_fold":k, **v} for k,v in best_params_by_fold.items()]))
print("Nested OOS compounded return:",
      float(np.prod(1.0 + outer_results["total_return"].dropna().to_numpy()) - 1.0)
      if outer_results["total_return"].notna().any() else np.nan)


C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\data\for_ml\builders\jerichon_feature_pipeline.py:77: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{c}_z{w}"] = (s - mean) / (std + 1e-12)
C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\data\for_ml\builders\jerichon_feature_pipeline.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{c}_ma{w}"] = mean
C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\data\for_ml\builders\jerichon_feature_pipeline.py:71: PerformanceWarning: DataFrame is highly fragmented.  Thi


=== WF1 | train ends=2026-09-28 12:24:49+00:00 | validation=2026-09-28 12:24:50+00:00 → 2026-10-05 08:07:06+00:00 ===


  0%|          | 0/5 [00:00<?, ?it/s]

C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\data\for_ml\builders\jerichon_feature_pipeline.py:77: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{c}_z{w}"] = (s - mean) / (std + 1e-12)
C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\data\for_ml\builders\jerichon_feature_pipeline.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{c}_ma{w}"] = mean
C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\data\for_ml\builders\jerichon_feature_pipeline.py:71: PerformanceWarning: DataFrame is highly fragmented.  Thi

[I 2026-10-18 23:21:12,287] Trial 0 finished with value: -0.7406986825639119 and parameters: {'dollar_threshold': 5000000, 'cusum_threshold': 0.00042673654483355207, 'tp_mult': 1.2, 'sl_mult': 0.7, 'vertical_horizon': 60, 'target_window': 60, 'confidence_threshold': 0.55}. Best is trial 0 with value: -0.7406986825639119.
[I 2026-10-18 23:21:15,293] Trial 1 finished with value: -0.04677442103348604 and parameters: {'dollar_threshold': 10000000, 'cusum_threshold': 0.0021840500188003097, 'tp_mult': 1.1, 'sl_mult': 1.3, 'vertical_horizon': 10, 'target_window': 90, 'confidence_threshold': 0.675}. Best is trial 1 with value: -0.04677442103348604.
[I 2026-10-18 23:21:22,274] Trial 2 finished with value: -0.014611950927578877 and parameters: {'dollar_threshold': 5000000, 'cusum_threshold': 0.00046836324999581627, 'tp_mult': 1.7000000000000002, 'sl_mult': 0.3, 'vertical_horizon': 100, 'target_window': 80, 'confidence_threshold': 0.725}. Best is trial 2 with value: -0.014611950927578877.
[I 2026

C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\data\for_ml\builders\jerichon_feature_pipeline.py:77: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{c}_z{w}"] = (s - mean) / (std + 1e-12)
C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\data\for_ml\builders\jerichon_feature_pipeline.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{c}_ma{w}"] = mean
C:\Users\Сергей Кравцов\Desktop\Morgan Co\JERICHON\data\for_ml\builders\jerichon_feature_pipeline.py:71: PerformanceWarning: DataFrame is highly fragmented.  Thi

... (8 outputs left out)

[I 2026-10-18 23:24:39,499] Trial 0 finished with value: -0.7948410364713421 and parameters: {'dollar_threshold': 5000000, 'cusum_threshold': 0.00042673654483355207, 'tp_mult': 1.2, 'sl_mult': 0.7, 'vertical_horizon': 60, 'target_window': 60, 'confidence_threshold': 0.55}. Best is trial 0 with value: -0.7948410364713421.
[I 2026-10-18 23:24:42,658] Trial 1 finished with value: -0.04677442103348604 and parameters: {'dollar_threshold': 10000000, 'cusum_threshold': 0.0021840500188003097, 'tp_mult': 1.1, 'sl_mult': 1.3, 'vertical_horizon': 10, 'target_window': 90, 'confidence_threshold': 0.675}. Best is trial 1 with value: -0.04677442103348604.
[I 2026-10-18 23:24:51,208] Trial 2 finished with value: -0.029353459717021836 and parameters: {'dollar_threshold': 5000000, 'cusum_threshold': 0.00046836324999581627, 'tp_mult': 1.7000000000000002, 'sl_mult': 0.3, 'vertical_horizon': 100, 'target_window': 80, 'confidence_threshold': 0.725}. Best is trial 2 with value: -0.029353459717021836.
[I 2026

,fold,dollar_threshold,cusum_threshold,tp_mult,sl_mult,vertical_horizon,target_window,confidence_threshold,total_return,trades,win_rate,profit_factor,max_drawdown,balanced_accuracy,roc_auc,brier,status
0,WF1,5000000,0.000468,1.7,0.3,100,80,0.725,0.0,0,NaN,0.0,0.0,0.500956,0.521542,0.198887,OK
1,WF2,5000000,0.000468,1.7,0.3,100,80,0.725,0.0,0,NaN,0.0,0.0,0.500000,0.531412,0.179184,OK
2,WF3,5000000,0.000468,1.7,0.3,100,80,0.725,0.0,0,NaN,0.0,0.0,0.500000,0.516160,0.190448,OK


,outer_fold,dollar_threshold,cusum_threshold,tp_mult,sl_mult,vertical_horizon,target_window,confidence_threshold
0,WF1,5000000,0.000468,1.7,0.3,100,80,0.725
1,WF2,5000000,0.000468,1.7,0.3,100,80,0.725
2,WF3,5000000,0.000468,1.7,0.3,100,80,0.725


Nested OOS compounded return: 0.0


## 5. Обязательное сравнение с фиксированным baseline

Без baseline нельзя сказать, помогла ли Optuna. Ниже фиксированная исходная конфигурация (CUSUM/TP/SL/horizon из текущего notebook, confidence 0.65) оценивается на тех же внешних календарных folds и тем же RF/исполнением. Это не участвует в оптимизации.


In [60]:
def evaluate_config_on_outer(outer, params, tag):
    activate_dollar_threshold(params.get("dollar_threshold", DOLLAR_THRESHOLD_FIXED))
    built = build_trial_dataset(
        params["cusum_threshold"], params["tp_mult"], params["sl_mult"],
        params["vertical_horizon"], params["target_window"]
    )
    if built is None:
        return {"fold":outer["name"], "configuration":tag, "total_return":np.nan,
                "trades":0, "status":"NO_DATA"}
    X, y, labels = built
    tr_idx = X.index[X.index <= outer["train_end_ts"]]
    va_idx = X.index[(X.index >= outer["val_start"]) & (X.index <= outer["val_end"])]
    va_idx = va_idx[(labels.reindex(va_idx)["t1"] <= outer["val_end"]).fillna(False).to_numpy()]
    if len(tr_idx) < 80 or len(va_idx) < 20:
        return {"fold":outer["name"], "configuration":tag, "total_return":np.nan,
                "trades":0, "status":"TOO_FEW_EVENTS"}
    fold = {"name":outer["name"], "train_candidate":tr_idx, "val_idx":va_idx,
            "val_start":outer["val_start"], "val_end":outer["val_end"]}
    result = eval_model_fold(fold, X, y, labels, params["vertical_horizon"], params["confidence_threshold"])
    if result is None:
        return {"fold":outer["name"], "configuration":tag, "total_return":np.nan,
                "trades":0, "status":"FOLD_SKIPPED"}
    return {
        "fold":outer["name"], "configuration":tag, **params,
        "total_return":result["total_return"], "trades":result["trades"],
        "win_rate":result["win_rate"], "profit_factor":result["profit_factor"],
        "max_drawdown":result["max_drawdown"], "balanced_accuracy":result["balanced_accuracy"],
        "roc_auc":result["roc_auc"], "brier":result["brier"], "status":"OK",
    }

baseline_params = {
    "dollar_threshold":int(DOLLAR_THRESHOLD_FIXED),
    "dollar_threshold":int(DOLLAR_THRESHOLD_FIXED),
    "cusum_threshold":float(CUSUM_THRESHOLD),
    "tp_mult":float(PT_SL[0]), "sl_mult":float(PT_SL[1]),
    "vertical_horizon":int(VERTICAL_HORIZON),
    "target_window":int(TARGET_WINDOW_FIXED),
    "confidence_threshold":0.65,
}
baseline_rows = []
for outer in make_outer_fold_specs():
    baseline_rows.append(evaluate_config_on_outer(outer, baseline_params, "fixed_baseline"))
    chosen = best_params_by_fold.get(outer["name"])
    if chosen:
        baseline_rows.append(evaluate_config_on_outer(outer, chosen, "optuna_nested"))
baseline_comparison = pd.DataFrame(baseline_rows)
display(baseline_comparison[[
    c for c in ["fold","configuration","total_return","trades","win_rate","profit_factor",
                "max_drawdown","balanced_accuracy","roc_auc","brier","status"]
    if c in baseline_comparison.columns
]])


,fold,configuration,total_return,trades,win_rate,profit_factor,max_drawdown,balanced_accuracy,roc_auc,brier,status
0,WF1,fixed_baseline,0.000000,0,NaN,0.000000,0.000000,0.548527,0.563719,0.247315,OK
1,WF1,optuna_nested,0.000000,0,NaN,0.000000,0.000000,0.500956,0.521542,0.198887,OK
2,WF2,fixed_baseline,-0.145862,99,0.040404,0.003781,0.145862,0.502390,0.530104,0.262497,OK
3,WF2,optuna_nested,0.000000,0,NaN,0.000000,0.000000,0.500000,0.531412,0.179184,OK
4,WF3,fixed_baseline,0.000000,0,NaN,0.000000,0.000000,0.519105,0.527954,0.249108,OK
5,WF3,optuna_nested,0.000000,0,NaN,0.000000,0.000000,0.500000,0.516160,0.190448,OK


## 6. Проверяем прогностическую способность и confidence без самообмана

`confidence_threshold` не делает классификатор точнее сам по себе: он только выбирает подмножество прогнозов. Поэтому отдельно строим таблицу по корзинам confidence — сколько прогнозов попало в корзину, какая доля направлений совпала с Triple Barrier, и какой был результат стратегии. Если у confidence 0.60–0.65 точность около 50%, фильтр не создаёт edge; он лишь сокращает число сделок. Калибровка оценивается только на внешних OOS прогнозах.

In [61]:
# Re-evaluate the selected configuration on each untouched outer fold and retain OOS predictions/trades.
oos_prediction_rows, oos_trade_rows = [], []
activate_dollar_threshold(DOLLAR_THRESHOLD_FIXED)
baseline = build_trial_dataset(CUSUM_THRESHOLD, PT_SL[0], PT_SL[1], VERTICAL_HORIZON, TARGET_WINDOW_FIXED)
X0, y0, labels0 = baseline
outer_folds = make_outer_fold_specs()

for outer in outer_folds:
    params = best_params_by_fold.get(outer["name"])
    if not params:
        continue
    activate_dollar_threshold(params["dollar_threshold"])
    built = build_trial_dataset(params["cusum_threshold"], params["tp_mult"], params["sl_mult"],
                                params["vertical_horizon"], params["target_window"])
    if built is None:
        continue
    X, y, labels = built
    tr_idx = X.index[X.index <= outer["train_end_ts"]]
    va_idx = X.index[(X.index >= outer["val_start"]) & (X.index <= outer["val_end"])]
    va_idx = va_idx[(labels.reindex(va_idx)["t1"] <= outer["val_end"]).fillna(False).to_numpy()]
    tr_idx = purge_train_indices(tr_idx, labels, outer["val_start"]) if len(va_idx) else tr_idx
    if len(tr_idx) < 80 or len(va_idx) < 20 or y.loc[tr_idx].nunique() < 2:
        continue
    Xtr, Xva = prepare_xy(X.loc[tr_idx], y.loc[tr_idx], X.loc[va_idx])
    model = make_rf()
    t1 = labels.reindex(tr_idx)["t1"]
    duration = (t1 - pd.Series(tr_idx, index=tr_idx)).dt.total_seconds().clip(lower=1.0)
    weights = (1.0 / duration)
    weights = weights / weights.mean()
    model.fit(Xtr, y.loc[tr_idx], sample_weight=weights.to_numpy())
    p = pd.Series(model.predict_proba(Xva)[:,1], index=va_idx)
    pred_side = (p >= 0.5).astype(int)
    conf = np.maximum(p, 1-p)
    correct = (pred_side == y.loc[va_idx]).astype(int)
    for ts in va_idx:
        oos_prediction_rows.append({
            "fold":outer["name"], "timestamp":ts, "y_true":int(y.loc[ts]),
            "p_up":float(p.loc[ts]), "predicted_side":int(pred_side.loc[ts]),
            "confidence":float(conf.loc[ts]), "correct":int(correct.loc[ts]),
            "confidence_threshold":params["confidence_threshold"],
        })
    f = {"name":outer["name"], "train_candidate":tr_idx, "val_idx":va_idx,
         "val_start":outer["val_start"], "val_end":outer["val_end"]}
    bt = simulate_oos_fold(f, p, labels, params["vertical_horizon"], params["confidence_threshold"])
    for r in bt["trade_rows"]:
        r["fold"] = outer["name"]
        r["optuna_confidence_threshold"] = params["confidence_threshold"]
        oos_trade_rows.append(r)

oos_predictions = pd.DataFrame(oos_prediction_rows)
oos_trades = pd.DataFrame(oos_trade_rows)

if not oos_predictions.empty:
    oos_predictions["confidence_bin"] = pd.cut(
        oos_predictions["confidence"],
        bins=[0.50,0.55,0.60,0.65,0.70,0.75,0.80,1.000001],
        right=False, include_lowest=True
    )
    confidence_report = oos_predictions.groupby("confidence_bin", observed=False).agg(
        predictions=("correct","size"), empirical_accuracy=("correct","mean"),
        mean_confidence=("confidence","mean"), mean_p_up=("p_up","mean")
    ).reset_index()
    display(confidence_report)
    print("OOS balanced accuracy:", balanced_accuracy_score(oos_predictions["y_true"], oos_predictions["predicted_side"]))
    print("OOS ROC AUC:", roc_auc_score(oos_predictions["y_true"], oos_predictions["p_up"])
          if oos_predictions["y_true"].nunique()==2 else np.nan)
else:
    confidence_report = pd.DataFrame()
    print("No outer OOS predictions available; inspect outer_results/status.")
display(oos_trades.head(20))


,confidence_bin,predictions,empirical_accuracy,mean_confidence,mean_p_up
0,"[0.5, 0.55)",252,0.765873,0.535375,0.465610
1,"[0.55, 0.6)",1358,0.831370,0.579149,0.420851
2,"[0.6, 0.65)",1737,0.831318,0.624634,0.375366
3,"[0.65, 0.7)",625,0.835200,0.665674,0.334326
4,"[0.7, 0.75)",20,0.850000,0.709398,0.290602
5,"[0.75, 0.8)",0,NaN,NaN,NaN
6,"[0.8, 1.0)",0,NaN,NaN,NaN


OOS balanced accuracy: 0.5004088738471353
OOS ROC AUC: 0.5184617838062922


""


## 7. Итоговая диагностика и минимальный экспорт

В первую очередь проверяем сшитую внешнюю OOS-доходность, результат каждого WF, количество сделок, drawdown и таблицу confidence. Нулевое число сделок на внешнем fold не является нулевой доходностью стратегии в общем смысле — это означает, что выбранный порог не дал сигналов в данном периоде; поэтому такие folds нужно отдельно видеть в отчёте.

Параметры, выбранные на разных outer folds, могут отличаться. Это диагностический признак нестабильности. Не следует просто взять самый прибыльный fold или лучший trial и объявить его готовой стратегией: это будет selection bias.

Подробный trade audit остаётся в памяти (`oos_trades`); большой CSV автоматически не пишется.


In [62]:
# Compact exports only; detailed trades remain available in-memory for focused analysis.
if not outer_results.empty:
    valid_returns = outer_results["total_return"].dropna().to_numpy(dtype=float)
    stitched_return = float(np.prod(1.0 + valid_returns) - 1.0) if len(valid_returns) else np.nan
    compact_summary = pd.DataFrame([{
        "ticker":TICKER,
        "dollar_threshold_candidates":str(DOLLAR_THRESHOLD_CANDIDATES),
        "outer_folds_evaluated":int(outer_results["total_return"].notna().sum()),
        "stitched_oos_total_return":stitched_return,
        "mean_fold_return":float(np.mean(valid_returns)) if len(valid_returns) else np.nan,
        "worst_fold_return":float(np.min(valid_returns)) if len(valid_returns) else np.nan,
        "total_oos_trades":int(outer_results["trades"].fillna(0).sum()),
        "mean_fold_drawdown":float(outer_results["max_drawdown"].mean()) if "max_drawdown" in outer_results else np.nan,
        "n_optuna_trials_per_fold":N_TRIALS_PER_OUTER_FOLD,
        "rf_parameters_fixed":f"n_estimators={RF_N_ESTIMATORS}; max_depth={RF_MAX_DEPTH}; min_samples_leaf={RF_MIN_SAMPLES_LEAF}",
    }])
    display(compact_summary)
    out_dir = ROOT.parent / "research" / "afml" / "afml_04_optuna" / "results"
    out_dir.mkdir(parents=True, exist_ok=True)
    compact_summary.to_csv(out_dir / "optuna_oos_summary.csv", index=False)
    outer_results.to_csv(out_dir / "optuna_outer_folds.csv", index=False)
    if "baseline_comparison" in globals():
        baseline_comparison.to_csv(out_dir / "optuna_baseline_comparison.csv", index=False)
    # No trade-audit CSV by default. Save manually only if a trade-level investigation is planned:
    # oos_trades.to_csv(out_dir / "optuna_oos_trades.csv", index=False)
    print("Saved compact outputs to:", out_dir.resolve())
    print("Trade audit rows in memory:", len(oos_trades))


,ticker,dollar_threshold_candidates,outer_folds_evaluated,stitched_oos_total_return,mean_fold_return,worst_fold_return,total_oos_trades,mean_fold_drawdown,n_optuna_trials_per_fold,rf_parameters_fixed
0,VTBR,"[5000000, 7500000, 10000000, 15000000]",3,0.0,0.0,0.0,0,0.0,5,n_estimators=350; max_depth=8; min_samples_leaf=8


Saved compact outputs to: C:\Users\Сергей Кравцов\afml_optuna_outputs
Trade audit rows in memory: 0
